# The `new` Operator

`new` creates an instance of a user-defined object type, or of a built-in type that has a constructor. It is the bridge between a function and the prototype system.

Related: [[JS - Functional Constructors and Errors]]

---

## 1. What `new` Does Under the Hood

Four sequential steps:

1. **Creates a blank object** — a plain `{}`.
2. **Links the prototype** — sets the new object's internal `[[Prototype]]` (exposed as `__proto__`) to the constructor's `.prototype` property, so the object inherits its methods.
3. **Binds `this`** — runs the constructor body with the new object as `this`.
4. **Returns the object** — automatically, unless the constructor explicitly returns its own non-primitive.

### Writing it yourself

The clearest way to internalise those steps is to implement them:

```js
function myNew(Ctor, ...args) {
  // 1 + 2: fresh object, prototype linked
  const obj = Object.create(Ctor.prototype);

  // 3: run the constructor with 'this' bound
  const result = Ctor.apply(obj, args);

  // 4: explicit object return wins, otherwise return obj
  const isObject = (result !== null && typeof result === 'object')
                || typeof result === 'function';
  return isObject ? result : obj;
}

myNew(User, 'Alice', 'Admin');  // behaves like new User('Alice', 'Admin')
```

The only thing this misses is `new.target`, which cannot be faked from userland — that's what `Reflect.construct` is for (see §5).

---

## 2. Code Examples

### With a class (modern)

```js
class SmartPhone {
  constructor(brand, model) {
    this.brand = brand;
    this.model = model;
  }

  getDetails() {
    return `${this.brand} ${this.model}`;
  }
}

const myPhone = new SmartPhone('Apple', 'iPhone 15');
myPhone.getDetails();  // "Apple iPhone 15"
```

`getDetails` lives on `SmartPhone.prototype`, not on `myPhone` — one shared function for every instance. Only `brand` and `model` are own properties.

### With a constructor function (pre-ES6)

```js
function User(name, role) {
  this.name = name;
  this.role = role;
}
User.prototype.describe = function () {
  return `${this.name} (${this.role})`;
};

const admin = new User('Alice', 'Admin');
admin.name;  // "Alice"
```

Structurally identical. Classes are sugar over this, not a separate object model.

### Without `new` at all — the factory alternative

```js
function createUser(name, role) {
  return {
    name,
    role,
    describe() { return `${name} (${role})`; }
  };
}

const admin = createUser('Alice', 'Admin');  // no 'new' needed
```

Factories use closures instead of prototypes. No `this`, no `new`, no missing-`new` bug, and genuinely private variables. The trade-off is that methods are re-created per object and `instanceof` doesn't work. Worth knowing as a real alternative rather than a lesser one.

---

## 3. Key Rules & Behaviours

### Explicit returns

- Return a **primitive** (string, number, boolean, `undefined`, `null`, symbol) → ignored, `this` is returned.
- Return an **object or function** → that value overrides `this` entirely.

```js
function A() { this.x = 1; return { x: 99 }; }
function B() { this.x = 1; return 42; }

new A().x;  // 99
new B().x;  // 1
```

This applies to class constructors too, and it's the mechanism behind singleton and caching patterns.

### `new.target`

Inside any function, `new.target` is the constructor that `new` was invoked on, or `undefined` for a plain call.

```js
function User(name) {
  if (!new.target) throw new TypeError("User must be called with 'new'");
  this.name = name;
}
```

In a **derived** class constructor, `new.target` is the class that was originally instantiated, not the one currently executing. That gives you a clean abstract-class guard:

```js
class Shape {
  constructor() {
    if (new.target === Shape) {
      throw new TypeError('Shape is abstract and cannot be instantiated');
    }
  }
}
class Circle extends Shape {}

new Circle();  // fine — new.target is Circle
new Shape();   // TypeError
```

### What cannot be constructed

Calling `new` on these throws `TypeError: X is not a constructor`:

- Arrow functions — no `[[Construct]]` internal method, no own `this`
- Shorthand methods, in object literals *and* in classes: `{ foo() {} }`
- Getters and setters
- Generator functions and async functions
- `Symbol` and `BigInt` — deliberately, to stop wrapper-object confusion

Only functions with a `[[Construct]]` slot are constructable. Ordinary `function` declarations and expressions have one; the list above does not.

### Classes must use `new`

```js
class Foo {}
Foo();  // TypeError: Class constructor Foo cannot be invoked without 'new'
```

Unlike function constructors, this is enforced by the language. It's the single biggest practical argument for classes.

---

## 4. Gotchas

### Operator precedence: `new Foo()` vs `new Foo`

Both work, but they parse differently. `new` *with* an argument list binds tighter than `new` without one.

```js
new Date().getTime();   // (new Date()).getTime()  ✅
new Date.now();         // new (Date.now)()  →  TypeError, Date.now isn't a constructor
```

Always include the parentheses. When the constructor itself comes from an expression, wrap it:

```js
new (getConstructor())(arg);
new (map.get('User'))(arg);
```

### Wrapper objects with built-ins

```js
typeof new String('hi');   // "object"  — not "string"
new String('hi') === 'hi'; // false
new Boolean(false) ? 'yes' : 'no';  // "yes" — every object is truthy
```

Never use `new` with `String`, `Number`, or `Boolean`. Call them as plain functions for conversion: `Number('42')`, `String(42)`.

`new Array(3)` is its own trap — a single numeric argument sets the *length*, producing three empty slots rather than `[3]`. Use `Array.of(3)` or a literal.

### Bound functions

`bind` fixes `this` for normal calls, but `new` ignores that binding. Pre-bound *arguments* still apply, and the prototype chain still points at the original.

```js
function Point(x, y) { this.x = x; this.y = y; }
const XAxis = Point.bind(null, 0);

const p = new XAxis(5);
p;                      // { x: 0, y: 5 }  — bound arg kept, bound 'this' discarded
p instanceof Point;     // true
```

### `this` is in TDZ before `super()`

In a derived class constructor, `this` doesn't exist until `super()` has run:

```js
class Circle extends Shape {
  constructor(r) {
    this.r = r;   // ReferenceError: Must call super constructor first
    super();
  }
}
```

Class fields are initialised immediately after `super()` returns — which is why a field can overwrite something the parent constructor set.

---

## 5. `Reflect.construct` — `new` as a function

```js
Reflect.construct(target, argsArray, newTarget);
```

Equivalent to `new target(...args)`, but with two extras: arguments come as an array, and you can decouple `new.target` from the function being run. The prototype of the result comes from `newTarget.prototype`.

```js
class Base { constructor() { this.tag = new.target.name; } }
class Sub {}

Reflect.construct(Base, [], Sub).tag;  // "Sub"
```

This is the standard workaround for subclassing built-ins (`Error`, `Array`) when targeting ES5, since transpiled `super()` can't do it properly.

### Testing constructability

```js
function isConstructor(fn) {
  try {
    Reflect.construct(function () {}, [], fn);
    return true;
  } catch {
    return false;
  }
}
```

---

## References

- [MDN — new operator](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/new)
- [MDN — new.target](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/new.target)
- [MDN — Reflect.construct](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Reflect/construct)
- [MDN — Inheritance and the prototype chain](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Inheritance_and_the_prototype_chain)
- [MDN — Object.create](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Object/create)